In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 2. Local Markdown Knowledge Bot

## What We're Building
A knowledge assistant that ingests your **Markdown files** (notes, docs, wikis)
and lets you query them using natural language — all running locally.

**You will learn:**
- Using LlamaIndex for document indexing
- Markdown-specific parsing and chunking
- Building a query engine with local Ollama models
- Difference between LlamaIndex and LangChain approaches

**Stack:** Ollama, LlamaIndex, local vector store

In [ ]:
# Install dependencies (uncomment to run)
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

import warnings
warnings.filterwarnings("ignore")
print("Ready!")

## Step 1 — Create Sample Markdown Knowledge Base

We'll create a small collection of markdown files to simulate a personal
knowledge base or documentation set.

In [ ]:
from pathlib import Path
import os

# Create sample markdown knowledge base
KB_DIR = Path("sample_kb")
KB_DIR.mkdir(exist_ok=True)

# File 1: Python basics
(KB_DIR / "python_basics.md").write_text("""# Python Basics

## Variables and Types
Python is dynamically typed. Common types include:
- `int`: Integer numbers (42)
- `float`: Decimal numbers (3.14)
- `str`: Text strings ("hello")
- `bool`: Boolean values (True/False)
- `list`: Ordered mutable collections
- `dict`: Key-value mappings

## Functions
Functions are defined with `def`:
```python
def greet(name: str) -> str:
    return f"Hello, {name}!"
```

## List Comprehensions
A concise way to create lists:
```python
squares = [x**2 for x in range(10)]
evens = [x for x in range(20) if x % 2 == 0]
```
""")

# File 2: Git workflow
(KB_DIR / "git_workflow.md").write_text("""# Git Workflow Guide

## Branch Strategy
- `main`: Production-ready code
- `develop`: Integration branch
- `feature/*`: New features
- `hotfix/*`: Emergency fixes

## Common Commands
```bash
git checkout -b feature/new-thing    # Create branch
git add -A                           # Stage all changes
git commit -m "feat: add widget"     # Commit
git push origin feature/new-thing    # Push
git pull --rebase origin main        # Rebase on main
```

## Commit Message Convention
Use conventional commits:
- `feat:` — New feature
- `fix:` — Bug fix
- `docs:` — Documentation
- `refactor:` — Code restructuring
- `test:` — Adding tests
""")

# File 3: Docker basics
(KB_DIR / "docker_notes.md").write_text("""# Docker Notes

## Key Concepts
- **Image**: A read-only template with instructions for creating a container
- **Container**: A runnable instance of an image
- **Dockerfile**: Script to build an image
- **Volume**: Persistent data storage
- **Network**: Communication between containers

## Essential Commands
```bash
docker build -t myapp .              # Build image
docker run -d -p 8080:80 myapp       # Run container
docker ps                            # List running containers
docker logs <container_id>           # View logs
docker-compose up -d                 # Start multi-container app
```

## Dockerfile Best Practices
1. Use specific base image tags (not `latest`)
2. Minimize layers — combine RUN commands
3. Use `.dockerignore` to exclude unnecessary files
4. Run as non-root user for security
5. Use multi-stage builds for smaller images
""")

print(f"Created {len(list(KB_DIR.glob('*.md')))} markdown files in {KB_DIR}/")
for f in KB_DIR.glob("*.md"):
    print(f"  - {f.name} ({f.stat().st_size} bytes)")

## Step 2 — Set Up LlamaIndex with Ollama

LlamaIndex (formerly GPT Index) is designed specifically for building search and
retrieval systems over your data. It handles indexing, retrieval, and query
processing with less boilerplate than LangChain for document QA tasks.

In [ ]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    StorageContext,
)
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

# Configure LlamaIndex to use local Ollama models
Settings.llm = Ollama(
    model="qwen3:8b",
    base_url="http://localhost:11434",
    request_timeout=120.0,
    temperature=0.2,
)

Settings.embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
)

# Set chunk size for splitting
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print("LlamaIndex configured with local Ollama models!")

## Step 3 — Load and Index Markdown Files

In [ ]:
# Load all markdown files from the knowledge base directory
reader = SimpleDirectoryReader(
    input_dir=str(KB_DIR),
    required_exts=[".md"],
    recursive=True,
)

documents = reader.load_data()
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc.metadata.get('file_name', 'unknown')}: {len(doc.text)} chars")

# Build the vector index — this embeds all chunks locally
index = VectorStoreIndex.from_documents(documents, show_progress=True)
print("\nIndex built successfully!")

## Step 4 — Query Your Knowledge Base

In [ ]:
# Create a query engine
query_engine = index.as_query_engine(
    similarity_top_k=3,  # Retrieve top 3 most relevant chunks
)

# Ask questions
questions = [
    "How do I create a new git branch?",
    "What are the common Python data types?",
    "How do I build a Docker image?",
    "What is a list comprehension in Python?",
    "What commit message convention should I use?",
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    response = query_engine.query(q)
    print(f"A: {response}")
    # Show which source files were used
    if response.source_nodes:
        sources = set(n.metadata.get("file_name", "?") for n in response.source_nodes)
        print(f"  Sources: {', '.join(sources)}")

## Step 5 — Chat Mode with Memory

LlamaIndex supports chat mode which maintains conversation history,
allowing follow-up questions.

In [ ]:
from llama_index.core.chat_engine import CondenseQuestionChatEngine

# Create a chat engine with conversation memory
chat_engine = CondenseQuestionChatEngine.from_defaults(
    query_engine=query_engine,
    verbose=True,
)

# Multi-turn conversation
print("--- Turn 1 ---")
response1 = chat_engine.chat("Tell me about Docker volumes")
print(f"A: {response1}")

print("\n--- Turn 2 (follow-up) ---")
response2 = chat_engine.chat("How do I use them in a Dockerfile?")
print(f"A: {response2}")

print("\n--- Turn 3 (follow-up) ---")
response3 = chat_engine.chat("What about networking?")
print(f"A: {response3}")

In [ ]:
# Interactive function for your own questions
def ask_kb(question: str):
    """Query the markdown knowledge base."""
    response = query_engine.query(question)
    print(f"Answer: {response}")
    if response.source_nodes:
        print("\nSources:")
        for node in response.source_nodes:
            print(f"  - {node.metadata.get('file_name', '?')} (score: {node.score:.3f})")
            print(f"    Preview: {node.text[:100]}...")
    return str(response)

# Try it with your own question
ask_kb("What are Docker best practices?")

## Summary & Next Steps

**What you learned:**
- ✅ LlamaIndex document loading with `SimpleDirectoryReader`
- ✅ Building a `VectorStoreIndex` with local embeddings
- ✅ Query engines vs chat engines
- ✅ Conversation memory with `CondenseQuestionChatEngine`
- ✅ Source attribution for retrieved answers

**LlamaIndex vs LangChain:**
| Feature | LlamaIndex | LangChain |
|---------|-----------|-----------|
| Focus | Data indexing & retrieval | General LLM orchestration |
| RAG setup | Very concise | More manual |
| Flexibility | Opinionated, fewer choices | Very flexible |
| Best for | Document QA | Complex agent workflows |

**Next steps:** Try adding your own markdown notes to `sample_kb/` and re-index!

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

# Set LlamaIndex to use local Ollama
Settings.llm = Ollama(model="qwen3.5:9b", request_timeout=120.0)
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

# documents = SimpleDirectoryReader('data').load_data()
# index = VectorStoreIndex.from_documents(documents)
# query_engine = index.as_query_engine()
# response = query_engine.query("What are the main insights?")
print("LlamaIndex configured for local RAG execution!")
